In [1]:
!pip install pyspark -q

In [2]:
import os
import shutil

# Create Data Lake folders
os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver/preprocessed_sales", exist_ok=True)

# Copy the original dataset into Bronze
shutil.copy("sales.csv", "data_lake/bronze/sales.csv")

print("Data Lake structure created successfully!")

Data Lake structure created successfully!


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Data Mining Data Lake Assignment") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.0.4


In [4]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data_lake/bronze/sales.csv")

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [5]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [6]:
df.show(10, truncate=False)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12              |UPI             |P

In [7]:
print("Total record count:", df.count())

Total record count: 1000


In [8]:
from pyspark.sql.functions import col, sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [9]:
total_rows = df.count()
distinct_rows = df.distinct().count()

print("Total rows:", total_rows)
print("Distinct rows:", distinct_rows)
print("Exact duplicate rows:", total_rows - distinct_rows)

Total rows: 1000
Distinct rows: 990
Exact duplicate rows: 10


In [10]:
print("Invalid quantity:")
df.filter(col("quantity") <= 0).show()

print("Invalid unit price:")
df.filter(col("unit_price") < 0).show()

print("Invalid discount percentage:")
df.filter(
    (col("discount_percent") < 0) |
    (col("discount_percent") > 100)
).show()

Invalid quantity:
+--------+-----------+-------------+--------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name|             product|       category|quantity|unit_price|discount_percent|  payment_method|      city|         state|order_date|order_status|
+--------+-----------+-------------+--------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|  100114|      C0252|Keerthi Singh|         Ceiling Fan|Home Appliances|       0|      2920|              15|     Net Banking|   Chennai|    Tamil Nadu|2025-02-15|   Delivered|
|  100143|      C0184|   Aman Joshi|            Football|         Sports|       0|      1030|              15|      Debit Card|Coimbatore|    Tamil Nadu|2025-11-26|   Delivered|
|  100198|      C0014| Arjun Pillai|              Hoodie|       Clothing|      -2|      2160

In [11]:
from pyspark.sql.functions import col, trim, lower, count

# Check text fields for whitespace/capitalization inconsistencies
text_columns = [
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_status"
]

for c in text_columns:
    print(f"\n===== {c} =====")
    df.groupBy(col(c)).count().orderBy(col(c)).show(30, truncate=False)


===== customer_name =====
+----------------+-----+
|customer_name   |count|
+----------------+-----+
|NULL            |10   |
|Aarav Agarwal   |2    |
|Aarav Chopra    |3    |
|Aarav Iyer      |2    |
|Aarav Kulkarni  |5    |
|Aarav Kumar     |2    |
|Aarav Nair      |2    |
|Aarav Patel     |6    |
|Aarav Singh     |4    |
|Aarav Tripathi  |2    |
|Abhinav Gupta   |6    |
|Abhinav Kulkarni|3    |
|Abhinav Malhotra|4    |
|Abhinav Pandey  |2    |
|Abhinav Pillai  |6    |
|Abhinav Rao     |6    |
|Abhinav Yadav   |4    |
|Aditi Agarwal   |1    |
|Aditi Chopra    |6    |
|Aditi Kapoor    |2    |
|Aditi Mishra    |3    |
|Aditi Patel     |9    |
|Aditi Tripathi  |1    |
|Aditya Chopra   |5    |
|Aditya Malhotra |2    |
|Aditya Mehta    |2    |
|Aditya Naidu    |1    |
|Aditya Rao      |3    |
|Akash Jain      |1    |
|Akash Jain      |1    |
+----------------+-----+
only showing top 30 rows

===== product =====
+----------------------+-----+
|product               |count|
+--------------

In [16]:
from pyspark.sql.functions import col, try_to_timestamp, lit

date_check = df.withColumn(
    "parsed_order_date",
    try_to_timestamp(col("order_date"), lit("yyyy-MM-dd"))
)

print("Invalid/unparseable dates:")

date_check.filter(
    col("order_date").isNotNull() &
    col("parsed_order_date").isNull()
).select(
    "order_id",
    "order_date"
).show(50, truncate=False)

Invalid/unparseable dates:
+--------+----------+
|order_id|order_date|
+--------+----------+
|100097  |31/02/2026|
|100358  |2025/07/14|
|100448  |not-a-date|
|100634  |2025-02-30|
|100637  |2026-13-05|
|100969  |09-15-2026|
+--------+----------+



In [17]:
from pyspark.sql.functions import (
    col, trim, initcap, upper, regexp_replace,
    when, coalesce, to_date, try_to_timestamp, lit
)

# Start from the original Bronze DataFrame
df_clean = df.dropDuplicates()

print("Rows after removing exact duplicates:", df_clean.count())


# ---------------------------------------------------------
# 1. Clean text fields
# ---------------------------------------------------------

text_columns = [
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_status"
]

for c in text_columns:
    df_clean = df_clean.withColumn(
        c,
        initcap(trim(col(c)))
    )


# ---------------------------------------------------------
# 2. Handle missing text values
# ---------------------------------------------------------

for c in text_columns:
    df_clean = df_clean.withColumn(
        c,
        when(
            col(c).isNull() | (trim(col(c)) == ""),
            "Unknown"
        ).otherwise(col(c))
    )


# ---------------------------------------------------------
# 3. Validate quantity
#    Quantity must be greater than 0
# ---------------------------------------------------------

df_clean = df_clean.withColumn(
    "quantity",
    when(col("quantity") > 0, col("quantity"))
    .otherwise(None)
)


# ---------------------------------------------------------
# 4. Validate unit price
#    Unit price cannot be negative
# ---------------------------------------------------------

df_clean = df_clean.withColumn(
    "unit_price",
    when(col("unit_price") >= 0, col("unit_price"))
    .otherwise(None)
)


# ---------------------------------------------------------
# 5. Validate discount percentage
#    Valid range: 0 to 100
# ---------------------------------------------------------

df_clean = df_clean.withColumn(
    "discount_percent",
    when(
        (col("discount_percent") >= 0) &
        (col("discount_percent") <= 100),
        col("discount_percent")
    ).otherwise(None)
)


# ---------------------------------------------------------
# 6. Handle missing numerical values
# ---------------------------------------------------------

# Use median values for numerical columns
quantity_median = df_clean.approxQuantile(
    "quantity", [0.5], 0.01
)[0]

unit_price_median = df_clean.approxQuantile(
    "unit_price", [0.5], 0.01
)[0]

discount_median = df_clean.approxQuantile(
    "discount_percent", [0.5], 0.01
)[0]

df_clean = df_clean.fillna({
    "quantity": quantity_median,
    "unit_price": unit_price_median,
    "discount_percent": discount_median
})


# ---------------------------------------------------------
# 7. Clean and standardize order_date
# ---------------------------------------------------------

# Try the expected format first.
# Invalid dates become NULL instead of causing an error.

df_clean = df_clean.withColumn(
    "parsed_date",
    try_to_timestamp(
        col("order_date"),
        lit("yyyy-MM-dd")
    )
)

# Try a few common alternative formats
df_clean = df_clean.withColumn(
    "parsed_date",
    coalesce(
        col("parsed_date"),
        try_to_timestamp(col("order_date"), lit("yyyy/MM/dd")),
        try_to_timestamp(col("order_date"), lit("dd-MM-yyyy")),
        try_to_timestamp(col("order_date"), lit("MM-dd-yyyy")),
        try_to_timestamp(col("order_date"), lit("dd/MM/yyyy"))
    )
)

# Convert to consistent yyyy-MM-dd format
df_clean = df_clean.withColumn(
    "order_date",
    when(
        col("parsed_date").isNotNull(),
        col("parsed_date").cast("date")
    )
)

# Remove temporary column
df_clean = df_clean.drop("parsed_date")


# ---------------------------------------------------------
# 8. Display result
# ---------------------------------------------------------

print("Final rows after preprocessing:", df_clean.count())

df_clean.show(10, truncate=False)

Rows after removing exact duplicates: 990
Final rows after preprocessing: 990
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered  

In [18]:
print("Invalid quantity:")
df_clean.filter(col("quantity") <= 0).show()

print("Invalid unit price:")
df_clean.filter(col("unit_price") < 0).show()

print("Invalid discount:")
df_clean.filter(
    (col("discount_percent") < 0) |
    (col("discount_percent") > 100)
).show()

Invalid quantity:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+

Invalid unit price:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----

In [19]:
print("Rows with missing/invalid order dates:")

df_clean.filter(
    col("order_date").isNull()
).select(
    "order_id",
    "order_date"
).show(50, truncate=False)

Rows with missing/invalid order dates:
+--------+----------+
|order_id|order_date|
+--------+----------+
|100097  |NULL      |
|100637  |NULL      |
|100448  |NULL      |
|100634  |NULL      |
+--------+----------+



In [20]:
from pyspark.sql.functions import sum

print("NULL values after preprocessing:")

df_clean.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_clean.columns
]).show()

NULL values after preprocessing:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|            0|      0|       0|       0|         0|               0|             0|   0|    0|         4|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [21]:
print("Final row count:", df_clean.count())

print("\nFinal schema:")
df_clean.printSchema()

print("\nFinal sample:")
df_clean.show(10, truncate=False)

Final row count: 990

Final schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)


Final sample:
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+-------

In [22]:
import os
import shutil
import glob

# Temporary Spark output folder
temp_output = "silver_temp"

# Remove old output if it exists
if os.path.exists(temp_output):
    shutil.rmtree(temp_output)

# Write the cleaned DataFrame as a single CSV
df_clean.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(temp_output)

# Find the generated CSV file
csv_file = glob.glob(f"{temp_output}/part-*.csv")[0]

# Create the required silver folder
os.makedirs("silver", exist_ok=True)

# Copy/rename Spark's part file to the required filename
final_file = "silver/sales_silver.csv"
shutil.copy(csv_file, final_file)

print("Silver file created successfully!")
print("File:", final_file)
print("Size:", os.path.getsize(final_file), "bytes")

Silver file created successfully!
File: silver/sales_silver.csv
Size: 107544 bytes


In [23]:
import pandas as pd

silver_preview = pd.read_csv("silver/sales_silver.csv")

print("Rows:", len(silver_preview))
print("Columns:", len(silver_preview.columns))

silver_preview.head(10)

Rows: 990
Columns: 13


,order_id,customer_id,customer_name,product,category,quantity,unit_price,discount_percent,payment_method,city,state,order_date,order_status
0,100331,C0005,Sakshi Joshi,Football,Sports,2,670,30,Credit Card,Bengaluru,Karnataka,2025-06-06,Cancelled
1,100333,C0263,Siddharth Kulkarni,Programming Book,Books,4,670,0,Net Banking,Pune,Maharashtra,2025-05-08,Delivered
2,100506,C0228,Diya Tripathi,Competitive Exam Guide,Books,6,610,12,Upi,Coimbatore,Tamil Nadu,2025-06-02,Delivered
3,100931,C0204,Pranav Reddy,Dining Table,Furniture,10,37710,30,Credit Card,Pune,Maharashtra,2025-12-25,Cancelled
4,100168,C0179,Sai Sharma,Office Chair,Furniture,4,6820,35,Cash On Delivery,Coimbatore,Tamil Nadu,2026-06-04,Processing
5,100473,C0195,Tanvi Gupta,Badminton Racket,Sports,2,840,35,Debit Card,Kolkata,West Bengal,2026-03-24,Shipped
6,100532,C0149,Manish Pillai,Shampoo,Beauty,4,770,12,Cash On Delivery,Delhi,Delhi,2026-04-30,Delivered
7,100011,C0021,Keerthi Tripathi,Programming Book,Books,3,1140,35,Net Banking,Delhi,Delhi,2026-07-27,Delivered
8,100064,C0276,Ananya Agarwal,Microwave Oven,Home Appliances,3,16610,0,Upi,Hyderabad,Telangana,2025-05-15,Delivered
9,100176,C0138,Pranav Joshi,Electric Kettle,Home Appliances,2,2170,35,Net Banking,Vijayawada,Andhra Pradesh,2026-08-23,Delivered
